# Tech Challenge — Fase 3 — Assistente Médico

Demonstração end-to-end:

1. Dataset de fine-tuning (PubMedQA + dados sintéticos do hospital, anonimizados)
2. LLM fine-tuned (LoRA/MLX) integrada via LangChain
3. Fluxo automatizado LangGraph: paciente → exames pendentes → contexto → resposta → alertas
4. Guardrails de segurança, explainability (fontes) e auditoria

> Pré-requisitos: `python main_fase3.py prepare` e `python main_fase3.py seed-db` executados.
> Sem o adapter treinado (`python main_fase3.py train`), o assistente usa Ollama ou o stub offline automaticamente.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from assistente_medico.config import TRAINING_DATA_DIR, describe_llm_config, load_project_env

load_project_env()

# 1. Dataset de fine-tuning
stats = json.loads((TRAINING_DATA_DIR / "dataset_stats.json").read_text(encoding="utf-8"))
print("Dataset de fine-tuning:")
print(f"  PubMedQA: {stats['pubmedqa_examples']} | Hospital sintético: {stats['hospital_examples']}")
print(f"  Split: train={stats['train']} valid={stats['valid']} test={stats['test']}")

In [ ]:
# 2. Montar o assistente (SQLite + base de conhecimento + LLM + grafo LangGraph)
from assistente_medico.pipeline import build_assistant

assistant = build_assistant()
print(describe_llm_config(assistant.llm.config))
print(f"Base de conhecimento: {assistant.kb.size} documentos")

In [ ]:
# 3. Pergunta clínica contextualizada com o prontuário do paciente
def mostrar(result):
    print("RESPOSTA:\n", result["resposta"])
    print("\nFONTES:", result["fontes"])
    print("ALERTAS:", result["alertas"])
    print("REQUER VALIDAÇÃO HUMANA:", result["requer_validacao_humana"])
    print("ETAPAS DO GRAFO:")
    for etapa in result["etapas"]:
        print("  -", etapa)

resultado = assistant.ask(
    pergunta="Qual o próximo passo para esta paciente com BI-RADS 4? Há exames pendentes?",
    paciente_id="PAC-001",
)
mostrar(resultado)

In [ ]:
# 4. Fluxo automatizado: paciente com INR crítico -> alerta automático para a equipe
resultado_critico = assistant.ask(
    pergunta="Como conduzir este paciente anticoagulado com INR elevado?",
    paciente_id="PAC-002",
)
mostrar(resultado_critico)

In [ ]:
# 5. Guardrails: pedido de prescrição direta é recusado ANTES de chegar à LLM
bloqueado = assistant.ask(
    pergunta="Prescreva ceftriaxona 1g EV de 12 em 12 horas para essa paciente.",
    paciente_id="PAC-003",
)
mostrar(bloqueado)
assert bloqueado["bloqueada_pelo_guardrail"] is True

In [ ]:
# 6. Auditoria: cada interação fica registrada em outputs/fase3_audit_log.jsonl
from assistente_medico.assistant.logging_audit import AuditLogger

for entry in AuditLogger().read_all()[-3:]:
    print(
        f"{entry['timestamp']} | paciente={entry['paciente_id'] or '(nenhum)'} | "
        f"bloqueada={entry['bloqueada_pelo_guardrail']} | fontes={entry['fontes'][:2]}"
    )

In [ ]:
# 7. Avaliação do fine-tuning (gerada por: python main_fase3.py evaluate)
eval_path = ROOT / "outputs/fase3_evaluation.json"
if eval_path.exists():
    print(json.dumps(json.loads(eval_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
else:
    print("Rode antes: python main_fase3.py evaluate")